# Predict Customer Churn - Version 4

## Fixed Meta-Learner with LogisticRegression
- **Base Models**: XGBoost + LightGBM + CatBoost + LogisticRegression
- **Meta-Learner**: LogisticRegression (classifier, not Ridge regressor)
- **Training Data**: 601K rows (594K competition + 7K IBM)
- **Key Fix**: Replace Ridge with LogReg to avoid NaN issues
- **Output**: V7-V9 submissions with enhanced visualizations

## 1. Libraries & Setup

In [ ]:
import numpy as np
import pandas as pd
import os, warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('✅ All libraries loaded!')

## 2. Load All Datasets

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
original = pd.read_csv('/kaggle/input/datasets/thedrzee/customer-churn-in-telecom-sample-dataset-by-ibm/WA_Fn-UseC_-Telco-Customer-Churn.csv')

v1 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V1.csv')
v2 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V2.csv')

print(f'Train:    {train.shape}')
print(f'Test:     {test.shape}')
print(f'Original: {original.shape}')

# Fix & combine
original['Churn'] = original['Churn'].map({'Yes': 'Yes', 'No': 'No'})
original['id'] = range(900000, 900000 + len(original))
train_combined = pd.concat([train, original], ignore_index=True)
print(f'Combined: {train_combined.shape}')

## 3. Feature Engineering

In [ ]:
def preprocess_v4(df, is_train=True):
    df = df.copy()
    df.drop(columns=['id', 'customerID'], inplace=True, errors='ignore')
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
    
    if is_train and 'Churn' in df.columns:
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    
    for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
        if col in df.columns:
            df[col] = (df[col] == 'Yes').astype(int)
    df['gender'] = (df['gender'] == 'Male').astype(int)
    
    # Features (same as V3)
    df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'].clip(lower=1))
    df['ChargeRatio']      = df['MonthlyCharges'] / (df['AvgMonthlyCharge'].clip(lower=0.01))
    df['IsNewCustomer']    = (df['tenure'] <= 12).astype(int)
    df['IsLongTerm']       = (df['tenure'] >= 60).astype(int)
    df['ChargeXTenure']    = df['MonthlyCharges'] * df['tenure']
    df['IsHighValue']      = (df['MonthlyCharges'] > 70).astype(int)
    
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies']
    binned = {}
    for col in service_cols:
        if col in df.columns and df[col].dtype == object:
            binned[col + '_bin'] = (df[col] == 'Yes').astype(int)
    
    df['TotalServices'] = df['PhoneService'].copy()
    for col in service_cols:
        if col + '_bin' in binned:
            df['TotalServices'] += binned[col + '_bin']
    df['ChargePerService'] = df['MonthlyCharges'] / (df['TotalServices'].clip(lower=1))
    
    if 'Contract' in df.columns:
        contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
        df['ContractRisk'] = df['Contract'].map(contract_risk).fillna(2)
        df['TenureContractRisk']  = df['tenure'] * df['ContractRisk']
        df['ChargeContractRisk']  = df['MonthlyCharges'] * df['ContractRisk']
        df['HighRiskFlag'] = ((df['ContractRisk'] == 3) & 
                             (df['IsNewCustomer'] == 1)).astype(int)
    
    ohe_cols = ['MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies',
                'Contract', 'PaymentMethod']
    ohe_cols = [c for c in ohe_cols if c in df.columns]
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=int)
    
    return df

train_clean = preprocess_v4(train_combined, is_train=True)
test_clean  = preprocess_v4(test, is_train=False)

feature_cols = [c for c in train_clean.columns if c != 'Churn']
for col in feature_cols:
    if col not in test_clean.columns:
        test_clean[col] = 0
test_clean = test_clean[feature_cols]

print(f'Train clean: {train_clean.shape}')
print(f'Test clean:  {test_clean.shape}')

## 4. Setup — X, Y, CV

In [ ]:
y = train_clean['Churn']
X = train_clean.drop(columns=['Churn'])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

churn_ratio      = y.mean()
scale_pos_weight = (1 - churn_ratio) / churn_ratio

print(f'Training set: {X.shape}')
print(f'Churn rate:   {y.mean():.2%}')
print(f'Scale pos weight: {scale_pos_weight:.2f}')

## 5. Optuna Tuning — CatBoost (10 trials, faster)

In [ ]:
best_xgb_params = {
    'n_estimators': 967, 'max_depth': 5, 'learning_rate': 0.05002,
    'subsample': 0.7716, 'colsample_bytree': 0.6621,
    'min_child_weight': 9, 'reg_alpha': 5.67e-06, 'reg_lambda': 1.167, 'gamma': 0.5593
}

best_lgbm_params = {
    'n_estimators': 1049, 'max_depth': 7, 'learning_rate': 0.0271,
    'num_leaves': 127, 'min_child_samples': 14, 'subsample': 0.9054,
    'bagging_freq': 1, 'colsample_bytree': 0.9506,
    'reg_alpha': 0.5099, 'reg_lambda': 0.0144
}

def objective_catboost(trial):
    params = {
        'iterations':      trial.suggest_int('iterations', 300, 2000),
        'depth':           trial.suggest_int('depth', 3, 10),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg':     trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
    }
    model = CatBoostClassifier(**params, loss_function='Logloss',
                              auto_class_weights='Balanced', task_type='GPU',
                              random_state=42, verbose=False)
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print(' Tuning CatBoost (10 trials, faster)...')
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_catboost, n_trials=10, show_progress_bar=True)
print(f' Best CV ROC-AUC: {study_cb.best_value:.5f}')

## 6. Out-Of-Fold Generation — All Models

In [ ]:
print(' Generating OOF predictions...')

oof_xgb  = np.zeros(len(X))
oof_lgbm = np.zeros(len(X))
oof_cb   = np.zeros(len(X))
oof_lr   = np.zeros(len(X))
test_xgb = test_lgbm = test_cb = test_lr = 0  # Will accumulate

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # XGBoost
    xgb = XGBClassifier(**best_xgb_params, scale_pos_weight=scale_pos_weight,
                       objective='binary:logistic', eval_metric='auc',
                       tree_method='hist', device='cuda', random_state=42, verbosity=0)
    xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb.predict_proba(X_va)[:, 1]
    if fold == 1:
        test_xgb = xgb.predict_proba(test_clean)[:, 1] / 5
    else:
        test_xgb += xgb.predict_proba(test_clean)[:, 1] / 5
    
    # LightGBM
    lgbm = LGBMClassifier(**best_lgbm_params, objective='binary', metric='auc',
                         device='cpu', n_jobs=-1, is_unbalance=True,
                         random_state=42, verbose=-1)
    lgbm.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = lgbm.predict_proba(X_va)[:, 1]
    if fold == 1:
        test_lgbm = lgbm.predict_proba(test_clean)[:, 1] / 5
    else:
        test_lgbm += lgbm.predict_proba(test_clean)[:, 1] / 5
    
    # CatBoost
    cb = CatBoostClassifier(**study_cb.best_params, loss_function='Logloss',
                           auto_class_weights='Balanced', task_type='GPU',
                           random_state=42, verbose=False)
    cb.fit(X_tr, y_tr)
    oof_cb[val_idx] = cb.predict_proba(X_va)[:, 1]
    if fold == 1:
        test_cb = cb.predict_proba(test_clean)[:, 1] / 5
    else:
        test_cb += cb.predict_proba(test_clean)[:, 1] / 5
    
    # LogReg (scaled)
    scaler = StandardScaler()
    X_tr_lr = X_tr.copy()
    X_va_lr = X_va.copy()
    X_tr_lr[scale_cols] = scaler.fit_transform(X_tr[scale_cols])
    X_va_lr[scale_cols] = scaler.transform(X_va[scale_cols])
    
    lr = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
    lr.fit(X_tr_lr, y_tr)
    oof_lr[val_idx] = lr.predict_proba(X_va_lr)[:, 1]
    if fold == 1:
        test_lr = lr.predict_proba(scaler.transform(test_clean[scale_cols]))[:, 1] / 5
    else:
        test_lr += lr.predict_proba(scaler.transform(test_clean[scale_cols]))[:, 1] / 5
    
    print(f'  Fold {fold}/5 ✓')

print(f'\nOOF scores: XGB={roc_auc_score(y, oof_xgb):.5f}  '
      f'LGBM={roc_auc_score(y, oof_lgbm):.5f}  '
      f'CB={roc_auc_score(y, oof_cb):.5f}  '
      f'LR={roc_auc_score(y, oof_lr):.5f}')

## 7. Meta-Learner Stacking (LogisticRegression, FIXED)

In [ ]:
# Stack OOF
oof_stack = np.column_stack([oof_xgb, oof_lgbm, oof_cb, oof_lr])
test_stack = np.column_stack([test_xgb, test_lgbm, test_cb, test_lr])

# Find best C parameter for LogReg meta-learner
best_c = 1.0
best_c_score = 0
c_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

print(' Testing LogReg C values...')
for c in c_values:
    meta_lr = LogisticRegression(C=c, random_state=42, max_iter=1000)
    meta_lr.fit(oof_stack, y)
    meta_preds = meta_lr.predict_proba(oof_stack)[:, 1]
    auc = roc_auc_score(y, meta_preds)
    print(f'   C={c:7.3f} → AUC: {auc:.5f}')
    if auc > best_c_score:
        best_c = c
        best_c_score = auc

print(f'\n✓ Best C: {best_c} (AUC: {best_c_score:.5f})')

# Final meta-learner
meta_model = LogisticRegression(C=best_c, random_state=42, max_iter=1000)
meta_model.fit(oof_stack, y)

meta_oof = meta_model.predict_proba(oof_stack)[:, 1]
meta_test = meta_model.predict_proba(test_stack)[:, 1]

meta_oof_auc = roc_auc_score(y, meta_oof)
print(f'\nMeta-Learner OOF AUC: {meta_oof_auc:.5f}')

# Print coefficients
print(f'\nMeta-Learner Coefficients:')
for name, coef in [('XGBoost', meta_model.coef_[0][0]),
                   ('LightGBM', meta_model.coef_[0][1]),
                   ('CatBoost', meta_model.coef_[0][2]),
                   ('LogReg', meta_model.coef_[0][3])]:
    print(f'  {name:10} {coef:7.4f}')

## 8. Full Data Refit with Multiple Seeds

In [ ]:
seeds = [42, 123, 456, 789, 1024, 2024, 3141, 5678, 9999, 7777]
print(f'Refitting with {len(seeds)} seeds for stability...')

test_final = np.zeros((len(test_clean), len(seeds)))

for i, seed in enumerate(seeds, 1):
    # Base models
    xgb_full = XGBClassifier(**best_xgb_params, scale_pos_weight=scale_pos_weight,
                            objective='binary:logistic', eval_metric='auc',
                            tree_method='hist', device='cuda',
                            random_state=seed, verbosity=0)
    xgb_full.fit(X, y)
    xgb_test_seed = xgb_full.predict_proba(test_clean)[:, 1]
    
    lgbm_full = LGBMClassifier(**best_lgbm_params, objective='binary', metric='auc',
                              device='cpu', n_jobs=-1, is_unbalance=True,
                              random_state=seed, verbose=-1)
    lgbm_full.fit(X, y)
    lgbm_test_seed = lgbm_full.predict_proba(test_clean)[:, 1]
    
    cb_full = CatBoostClassifier(**study_cb.best_params, loss_function='Logloss',
                                auto_class_weights='Balanced', task_type='GPU',
                                random_state=seed, verbose=False)
    cb_full.fit(X, y)
    cb_test_seed = cb_full.predict_proba(test_clean)[:, 1]
    
    scaler = StandardScaler()
    X_scaled = X.copy()
    test_scaled = test_clean.copy()
    X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])
    test_scaled[scale_cols] = scaler.transform(test_clean[scale_cols])
    
    lr_full = LogisticRegression(random_state=seed, max_iter=1000, C=1.0)
    lr_full.fit(X_scaled, y)
    lr_test_seed = lr_full.predict_proba(test_scaled)[:, 1]
    
    # Stack & meta-predict
    test_stack_seed = np.column_stack([xgb_test_seed, lgbm_test_seed, cb_test_seed, lr_test_seed])
    test_final[:, i-1] = meta_model.predict_proba(test_stack_seed)[:, 1]
    
    if i % 2 == 0:
        print(f'  {i}/{len(seeds)} seeds done')

test_ensemble = test_final.mean(axis=1)
print(f'\nTest ensemble ready: mean={test_ensemble.mean():.4f}')

## 9. Submission Generation

In [ ]:
# Load previous submissions
v1_probs = v1.set_index('id').reindex(test['id'])['Churn'].values
v2_probs = v2.set_index('id').reindex(test['id'])['Churn'].values

# V7: Meta-stack only
sub_v7 = pd.DataFrame({'id': test['id'], 'Churn': test_ensemble})
sub_v7.to_csv('submission_v7_meta_stack.csv', index=False)
print('✓ submission_v7_meta_stack.csv')

# V8: Weighted blend with previous best
blend_v8 = 0.65 * test_ensemble + 0.35 * v1_probs
blend_v8 = np.clip(blend_v8, 0, 1)
sub_v8 = pd.DataFrame({'id': test['id'], 'Churn': blend_v8})
sub_v8.to_csv('submission_v8_blend.csv', index=False)
print('✓ submission_v8_blend.csv')

# V9: Rank-based blend (most stable)
from scipy.stats import rankdata
def rank_blend(*arrays, weights):
    ranked = [rankdata(a) / len(a) for a in arrays]
    return sum(w * r for w, r in zip(weights, ranked))

blend_v9 = rank_blend(test_ensemble, v1_probs, weights=[0.7, 0.3])
sub_v9 = pd.DataFrame({'id': test['id'], 'Churn': blend_v9})
sub_v9.to_csv('submission_v9_rank.csv', index=False)
print('✓ submission_v9_rank.csv')

## 10. CV-LB Tracking & Model Comparison

In [ ]:
# Version tracking table
versions_summary = pd.DataFrame({
    'V#': ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9'],
    'Method': ['XGB baseline', 'LogReg baseline', 'Weighted Blend',
               'Blend+V1', 'Blend All', 'Rank Blend',
               'Meta-Stack (V4)', 'Meta+Blend', 'Meta+Rank'],
    'CV ROC-AUC': [0.91689, 0.91282, 0.91594,
                   0.91501, '--', '--',
                   f'{meta_oof_auc:.5f}', '--', '--'],
    'Public LB': ['--', '--', '--',
                  '--', '--', '--',
                  'ACTIVE', 'ACTIVE', 'STABLE'],
    'Notes': ['baseline', 'deprecated', 'simple ensemble',
              'improved', 'multi-blend', 'rank stable',
              '← LogReg meta!', '← best public?', '← best private?']
})
print(versions_summary.to_string(index=False))

print(f'\nKey improvement: Ridge (V3) NaN issue FIXED with LogReg (V4)')
print(f'V3 Ridge OOF AUC: 0.91254 (NaN in CV scoring)')
print(f'V4 LogReg OOF AUC: {meta_oof_auc:.5f} (valid, no NaN)')

## 11. ROC Curves Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

models_for_roc = [
    ('XGBoost OOF',   oof_xgb,   0.91730, '#4E79A7'),
    ('LightGBM OOF',  oof_lgbm,  0.91618, '#E15759'),
    ('CatBoost OOF',  oof_cb,    0.91530, '#59A14F'),
    ('LogReg OOF',    oof_lr,    0.91420, '#F28E2B'),
    ('Meta-Stack',    meta_oof,  meta_oof_auc, '#B07AA1'),
]

for name, probs, expected_auc, color in models_for_roc:
    fpr, tpr, _ = roc_curve(y, probs)
    auc = roc_auc_score(y, probs)
    lw = 3 if 'Meta' in name else 2
    ax.plot(fpr, tpr, lw=lw, color=color,
           label=f'{name:15} (AUC={auc:.5f})', zorder=10-lw)

ax.plot([0,1], [0,1], 'k--', lw=1.5, label='Random (AUC=0.500)', zorder=1)
ax.fill_between([0,1], [0,1], 1, alpha=0.05, color='gray')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curves — All Models (V4)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10, framealpha=0.95)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Prediction Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

submissions_dist = [
    ('V7: Meta-Stack',    test_ensemble, '#B07AA1'),
    ('V8: Blend',         blend_v8,      '#59A14F'),
    ('V9: Rank Blend',    blend_v9,      '#E15759'),
]

for ax, (name, probs, color) in zip(axes, submissions_dist):
    ax.hist(probs, bins=60, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(0.5, color='black', lw=2.5, linestyle='--', label='Threshold (0.5)')
    ax.axvline(probs.mean(), color='red', lw=2, linestyle=':', label=f'Mean={probs.mean():.4f}')
    
    ax.set_xlabel('Predicted Churn Probability', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title(f'{name}\nStd={probs.std():.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Statistics
    below_threshold = (probs < 0.5).sum()
    above_threshold = (probs >= 0.5).sum()
    ax.text(0.02, 0.98, f'{below_threshold} no-churn\\n{above_threshold} churn',
           transform=ax.transAxes, fontsize=9, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Prediction Distributions — Final Submissions (V7-V9)',
            fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Final Summary

In [ ]:
print('\n' + '='*75)
print('VERSION 4 — FIXED META-LEARNER (LOGISTICREGRESSION) — FINAL SUMMARY')
print('='*75)

print(f'\nTRAINING DATA:')
print(f'  Rows:            {len(X):,} (594K competition + 7K IBM)')
print(f'  Features:        {X.shape[1]}')
print(f'  Churn Positive:  {(y==1).sum():,} ({y.mean()*100:.1f}%)')
print(f'')
print(f'BASE MODEL OOF ROC-AUC:')
print(f'  XGBoost:         {roc_auc_score(y, oof_xgb):.5f} (Optuna pre-tuned)')
print(f'  LightGBM:        {roc_auc_score(y, oof_lgbm):.5f} (Optuna pre-tuned)')
print(f'  CatBoost:        {roc_auc_score(y, oof_cb):.5f} (Optuna 10 trials)')
print(f'  LogReg:          {roc_auc_score(y, oof_lr):.5f}')
print(f'')
print(f'META-LEARNER PERFORMANCE:')
print(f'  Model:           LogisticRegression(C={best_c})')
print(f'  OOF ROC-AUC:     {meta_oof_auc:.5f}  ✓ (No NaN issues!)')
print(f'')
print(f'META-LEARNER BLEND WEIGHTS:')
print(f'  XGBoost:         {meta_model.coef_[0][0]:.4f} (dominance)')
print(f'  LightGBM:        {meta_model.coef_[0][1]:.4f} (similar to XGB)')
print(f'  CatBoost:        {meta_model.coef_[0][2]:.4f} (diversity)')
print(f'  LogReg:          {meta_model.coef_[0][3]:.4f} (low weight, linear diversity)')
print(f'')
print(f'TEST SET PREDICTIONS (seed-averaged):')
print(f'  V7 Meta-Stack:   mean={test_ensemble.mean():.4f}  std={test_ensemble.std():.4f}')
print(f'  V8 Blend:        mean={blend_v8.mean():.4f}  std={blend_v8.std():.4f}')
print(f'  V9 Rank Blend:   mean={blend_v9.mean():.4f}  std={blend_v9.std():.4f}')
print(f'')
print(f'KEY FIXES vs V3:')
print(f'  ✓ Ridge regressor → LogisticRegression classifier')
print(f'  ✓ No more NaN issues in CV scoring')
print(f'  ✓ Added ROC curves visualization')
print(f'  ✓ Added prediction distribution histograms')
print(f'  ✓ Multi-seed averaging for stability')
print(f'')
print(f'SUBMISSION STRATEGY:')
print(f'  1. Submit V7 (meta-stack) — most optimized')
print(f'  2. Monitor public LB → compare with V9')
print(f'  3. If V7 > V9 → stable meta-learning worked')
print(f'  4. If V9 > V7 → rank stability preferred')
print(f'  5. Save both V7 & V9 for final submissions')
print('='*75)